# Agentic RAG with Ollama (100% FREE)

This notebook implements a **true Agentic RAG system** using:
- **Ollama** for embeddings and LLM (runs locally, completely free)
- **ChromaDB** for vector storage

## What Makes This "Agentic"?

Unlike simple RAG (Question → Retrieve → Answer), this system has autonomous capabilities:

1. 🧠 **Query Rewriting** - Agent improves vague queries for better retrieval
2. ⚖️ **Relevance Grading** - Agent judges if retrieved docs are actually relevant
3. 🎯 **Conditional Response** - Agent decides whether to answer or decline safely
4. 🔄 **Multi-Step Reasoning** - Agent follows: Rewrite → Retrieve → Grade → Answer

## Setup Steps:

1. Install Ollama from: https://ollama.ai
2. Run these commands in terminal:
   ```bash
   ollama pull llama3.2
   ollama pull nomic-embed-text
   ```
3. Run this notebook step by step!

---
## Step 1: Install Required Packages

In [30]:
!pip install -U langchain langchain-ollama langchain-chroma langchain-community chromadb pypdf

You should consider upgrading via the '/Users/I531807/Documents/Edureka/AGENTIC RAG/venv/bin/python3 -m pip install --upgrade pip' command.


## Step 2: Setup Project Paths

In [31]:
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
PDF_DIR = PROJECT_ROOT / "data" / "pdfs"
PERSIST_DIR = PROJECT_ROOT / "storage" / "chroma_db_ollama_agentic"

PDF_DIR.mkdir(parents=True, exist_ok=True)
PERSIST_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Project root:", PROJECT_ROOT)
print("✅ PDF directory:", PDF_DIR)
print("✅ Persist directory:", PERSIST_DIR)

✅ Project root: /Users/I531807/Documents/Edureka/AGENTIC RAG
✅ PDF directory: /Users/I531807/Documents/Edureka/AGENTIC RAG/data/pdfs
✅ Persist directory: /Users/I531807/Documents/Edureka/AGENTIC RAG/storage/chroma_db_ollama_agentic


## Step 3: Load PDFs

In [32]:
from langchain_community.document_loaders import PyPDFLoader

def load_pdfs_from_folder(pdf_dir: Path):
    all_docs = []
    pdf_files = list(pdf_dir.glob("*.pdf"))

    if not pdf_files:
        print(f"⚠️  No PDF files found in: {pdf_dir}")
        return all_docs

    for pdf_file in pdf_files:
        print(f"📄 Loading: {pdf_file.name}")
        loader = PyPDFLoader(str(pdf_file))
        docs = loader.load()
        all_docs.extend(docs)

    return all_docs

raw_docs = load_pdfs_from_folder(PDF_DIR)
print(f"\n✅ Total loaded pages: {len(raw_docs)}")

📄 Loading: Agentic_RAG_Advanced_Guide.pdf
📄 Loading: RAG_Introduction_and_Fundamentals.pdf

✅ Total loaded pages: 9


## Step 4: Split Documents into Chunks

In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

split_docs = text_splitter.split_documents(raw_docs)
print(f"✅ Total chunks created: {len(split_docs)}")

✅ Total chunks created: 17


## Step 5: Create Embeddings with Ollama (FREE!)

In [34]:
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

print("✅ Ollama embedding model ready (100% FREE!)")

✅ Ollama embedding model ready (100% FREE!)


## Step 6: Build ChromaDB Vector Store

**Note:** Using a different persist directory so we don't overwrite the simple RAG database.

In [35]:
from langchain_chroma import Chroma

COLLECTION_NAME = "agentic_rag_ollama_collection"

vectorstore = Chroma.from_documents(
    documents=split_docs,
    embedding=embedding_model,
    persist_directory=str(PERSIST_DIR),
    collection_name=COLLECTION_NAME
)

print("✅ Vector store created with FREE Ollama embeddings!")

✅ Vector store created with FREE Ollama embeddings!


## Step 7: Create Retriever

In [36]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("✅ Retriever ready (fetching top 3 relevant chunks)")

✅ Retriever ready (fetching top 3 relevant chunks)


## Step 8: Create LLM with Ollama (FREE!)

In [37]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0  # Deterministic for consistent testing
)

print("✅ Ollama LLM ready (100% FREE!)")

✅ Ollama LLM ready (100% FREE!)


## Step 9: Baseline Simple RAG (For Comparison)

First, let's implement a **simple RAG** to compare against the agentic version.

**Simple RAG Flow:**
```
Question → Retrieve → Answer
```

In [38]:
def format_context(docs):
    """Helper function to format retrieved documents into context."""
    return "\n\n".join(doc.page_content for doc in docs)

def simple_rag(query: str, retriever, llm):
    """
    Simple RAG: Question → Retrieve → Answer
    No agent capabilities.
    """
    docs = retriever.invoke(query)
    context = format_context(docs)

    prompt = f'''
You are a helpful assistant.
Answer the question using only the provided context.
If the answer is not in the context, say you do not have enough information.

Context:
{context}

Question:
{query}
'''

    response = llm.invoke(prompt)
    return {
        "query": query,
        "retrieved_docs": docs,
        "answer": response.content
    }

print("✅ Simple RAG function ready")
print("\n⏸️  We'll add the AGENTIC capabilities in the next cells...")

✅ Simple RAG function ready

⏸️  We'll add the AGENTIC capabilities in the next cells...


---

# 🤖 AGENTIC RAG SECTION

Now we add the three key agentic capabilities!

---

## Step 10: Query Rewriting (Agentic Capability #1)

The **agent autonomously improves vague queries** to make them more retrieval-friendly.

**Examples:**
- "tell me stuff" → "What are the main topics discussed in the documents?"
- "what does it say" → "What are the key points and findings in the text?"

This is the first decision the agent makes on its own!

In [39]:
def rewrite_query(query: str, llm):
    """
    🧠 AGENT CAPABILITY #1: Query Rewriting
    
    The agent autonomously improves the user's query to be more specific
    and retrieval-friendly.
    
    Args:
        query: Original user question (possibly vague)
        llm: Language model
        
    Returns:
        Improved, more specific query
    """
    prompt = f'''
You are helping improve document retrieval.
Rewrite the user's question so it is clearer, more specific, and retrieval-friendly.
Do not answer the question.
Only return the improved query.

Original question:
{query}
'''
    response = llm.invoke(prompt)
    return response.content.strip()

print("✅ Query Rewriting function ready")

✅ Query Rewriting function ready


### Test Query Rewriting

Let's see the agent improve a vague question!

In [40]:
# Test the agent's query rewriting capability
test_queries = [
    "tell me about the important stuff",
    "what does it say",
    "give me info"
]

print("🧠 Agent Query Rewriting in Action:\n")
print("="*80)

for original in test_queries:
    improved = rewrite_query(original, llm)
    print(f"\n📝 Original:  {original}")
    print(f"✨ Improved:  {improved}")
    print("-"*80)

🧠 Agent Query Rewriting in Action:


📝 Original:  tell me about the important stuff
✨ Improved:  "What are the key points or most relevant information about a particular topic?"
--------------------------------------------------------------------------------

📝 Original:  what does it say
✨ Improved:  "What does a specific document or text say?"
--------------------------------------------------------------------------------

📝 Original:  give me info
✨ Improved:  "Provide general information on a wide range of topics"
--------------------------------------------------------------------------------


---

## Step 11: Relevance Grading (Agentic Capability #2)

The **agent judges whether retrieved documents are relevant** to the question.

This prevents the system from answering when the retrieved context is weak or off-topic.

**Agent Decision:**
- **YES** → Retrieved docs are relevant, proceed to answer
- **NO** → Not enough relevant info, decline safely

In [41]:
def grade_relevance(query: str, docs, llm):
    """
    ⚖️ AGENT CAPABILITY #2: Relevance Grading
    
    The agent evaluates whether the retrieved documents are relevant
    enough to answer the user's question.
    
    Args:
        query: User's question
        docs: Retrieved documents
        llm: Language model
        
    Returns:
        "YES" if relevant, "NO" if not relevant
    """
    context = format_context(docs)

    prompt = f'''
You are judging whether the retrieved context contains relevant information to answer the user's question.

Question:
{query}

Retrieved context:
{context}

If the context contains ANY information that helps answer the question, reply: YES
If the context is completely unrelated or empty, reply: NO

Be lenient - if there's even partial relevant information, say YES.

Reply with only one word:
YES
or
NO
'''
    result = llm.invoke(prompt)
    decision = result.content.strip().upper()
    return "YES" if "YES" in decision else "NO"

print("✅ Relevance Grading function ready (IMPROVED - less strict)")

✅ Relevance Grading function ready (IMPROVED - less strict)


### Test Relevance Grading

Let's test with a relevant question and an irrelevant one!

In [42]:
# Test relevance grading with different questions
print("⚖️  Agent Relevance Grading in Action:\n")
print("="*80)

# Test 1: Relevant question
relevant_query = "What is Agentic RAG?"
docs_relevant = retriever.invoke(relevant_query)
grade_relevant = grade_relevance(relevant_query, docs_relevant, llm)

print(f"\n📝 Question: {relevant_query}")
print(f"🎯 Agent Decision: {grade_relevant}")
print(f"   (Expected: YES - this is in our PDFs)")

print("\n" + "-"*80)

# Test 2: Irrelevant question
irrelevant_query = "What is quantum physics?"
docs_irrelevant = retriever.invoke(irrelevant_query)
grade_irrelevant = grade_relevance(irrelevant_query, docs_irrelevant, llm)

print(f"\n📝 Question: {irrelevant_query}")
print(f"🎯 Agent Decision: {grade_irrelevant}")
print(f"   (Expected: NO - this is NOT in our PDFs)")
print("="*80)

⚖️  Agent Relevance Grading in Action:


📝 Question: What is Agentic RAG?
🎯 Agent Decision: YES
   (Expected: YES - this is in our PDFs)

--------------------------------------------------------------------------------

📝 Question: What is quantum physics?
🎯 Agent Decision: NO
   (Expected: NO - this is NOT in our PDFs)


---

## Step 12: Grounded Answer Generation (Agentic Capability #3)

The **agent generates answers strictly from retrieved evidence** - no hallucinations allowed.

If the evidence doesn't support an answer, the agent declines safely.

In [43]:
def generate_grounded_answer(query: str, docs, llm):
    """
    🎯 AGENT CAPABILITY #3: Grounded Answer Generation
    
    The agent generates an answer strictly from the retrieved context.
    No hallucinations allowed - only facts from the documents.
    
    Args:
        query: User's question
        docs: Retrieved documents
        llm: Language model
        
    Returns:
        Answer grounded in evidence, or safe fallback
    """
    context = format_context(docs)

    prompt = f'''
You are a careful AI assistant.

Use only the retrieved context below.
Do not invent facts.
If the answer is not supported by the context, say:
"I do not have enough information in the retrieved documents."

Retrieved context:
{context}

Question:
{query}
'''
    response = llm.invoke(prompt)
    return response.content

print("✅ Grounded Answer Generation function ready")

✅ Grounded Answer Generation function ready


---

## Step 13: Complete Agentic RAG Pipeline 🤖

Now we combine all three agentic capabilities into one intelligent system!

**Agentic Workflow:**
```
User Question
   ↓
🧠 Agent Rewrites Query (improve vague questions)
   ↓
📚 Retrieve Relevant Chunks
   ↓
⚖️  Agent Grades Relevance (judge if docs are relevant)
   ↓
🎯 Agent Decides:
   - YES → Generate Grounded Answer
   - NO  → Decline Safely
```

In [44]:
def agentic_rag(query: str, retriever, llm, verbose=True):
    """
    🤖 COMPLETE AGENTIC RAG PIPELINE
    
    Combines all three agentic capabilities:
    1. Query rewriting
    2. Relevance grading
    3. Conditional grounded response
    
    Args:
        query: User's question
        retriever: Vector store retriever
        llm: Language model
        verbose: Show agent's thinking process
        
    Returns:
        Dictionary with query, improved query, relevance, docs, and answer
    """
    if verbose:
        print("🤖 AGENT STARTING...\n")
        print("="*80)
    
    # Step 1: Agent rewrites the query
    if verbose:
        print("\n🧠 STEP 1: Query Rewriting")
        print("-"*80)
        print(f"Original:  {query}")
    
    improved_query = rewrite_query(query, llm)
    
    if verbose:
        print(f"Improved:  {improved_query}")
    
    # Step 2: Retrieve documents
    if verbose:
        print("\n📚 STEP 2: Retrieving Documents")
        print("-"*80)
    
    docs = retriever.invoke(improved_query)
    
    if verbose:
        print(f"Retrieved {len(docs)} chunks")
    
    # Step 3: Agent grades relevance
    if verbose:
        print("\n⚖️  STEP 3: Grading Relevance")
        print("-"*80)
    
    relevance = grade_relevance(improved_query, docs, llm)
    
    if verbose:
        print(f"Agent Decision: {relevance}")
    
    # Step 4: Conditional response based on relevance
    if relevance == "NO":
        if verbose:
            print("\n⚠️  STEP 4: Agent Declined")
            print("-"*80)
            print("Reason: Not enough relevant information")
            print("="*80)
            print("\n✅ AGENT COMPLETED\n")
        
        return {
            "original_query": query,
            "improved_query": improved_query,
            "relevance": relevance,
            "retrieved_docs": docs,
            "answer": "I do not have enough relevant information in the retrieved documents to answer this question."
        }
    
    # Step 5: Agent generates grounded answer
    if verbose:
        print("\n🎯 STEP 4: Generating Grounded Answer")
        print("-"*80)
    
    answer = generate_grounded_answer(improved_query, docs, llm)
    
    if verbose:
        print("Answer generated from evidence only")
        print("="*80)
        print("\n✅ AGENT COMPLETED\n")
    
    return {
        "original_query": query,
        "improved_query": improved_query,
        "relevance": relevance,
        "retrieved_docs": docs,
        "answer": answer
    }

print("✅ Complete Agentic RAG Pipeline ready!")

✅ Complete Agentic RAG Pipeline ready!


### Test the Complete Agentic RAG System

Let's see the agent in action with a vague question!

In [45]:
# Test with a vague question - watch the agent work!
result = agentic_rag(
    query="TELL me the diff between agentic rag and simple rag",
    retriever=retriever,
    llm=llm,
    verbose=True
)

print("\n" + "="*80)
print("📋 FINAL ANSWER:")
print("="*80)
print(result["answer"])

🤖 AGENT STARTING...


🧠 STEP 1: Query Rewriting
--------------------------------------------------------------------------------
Original:  TELL me the diff between agentic rag and simple rag
Improved:  What are the key differences between Agentic Rag and Simple Rag?

📚 STEP 2: Retrieving Documents
--------------------------------------------------------------------------------
Retrieved 3 chunks

⚖️  STEP 3: Grading Relevance
--------------------------------------------------------------------------------
Agent Decision: YES

🎯 STEP 4: Generating Grounded Answer
--------------------------------------------------------------------------------
Answer generated from evidence only

✅ AGENT COMPLETED


📋 FINAL ANSWER:
According to the retrieved context, the key differences between Agentic RAG and Traditional RAG are:

1. Retrieval Strategy:
	* Traditional RAG: Single-step, fixed approach
	* Agentic RAG: Multi-step, adaptive reasoning
2. Query Processing:
	* Traditional RAG: Direct query em

### 🔧 Debug: Check Relevance Grading

Let's see why the agent might be declining good questions.

In [ ]:
# Debug: Let's manually check what's being retrieved and graded
debug_query = "tell me about the main stuff in these files"
print(f"Original query: {debug_query}\n")

# Rewrite
improved = rewrite_query(debug_query, llm)
print(f"Improved query: {improved}\n")

# Retrieve
docs = retriever.invoke(improved)
print(f"Retrieved {len(docs)} documents\n")

# Show retrieved content
print("="*80)
print("Retrieved Content:")
print("="*80)
for i, doc in enumerate(docs, 1):
    print(f"\nChunk {i}:")
    print(doc.page_content[:300])
    print("...")

# Grade
print("\n" + "="*80)
print("Grading Relevance...")
print("="*80)
grade = grade_relevance(improved, docs, llm)
print(f"\nAgent Decision: {grade}")
print("\n⚠️  If this is 'NO' for a good question, we need to fix the relevance grading!")

### Test with an Irrelevant Question

Watch the agent decline when there's no relevant information!

In [46]:
# Test with an irrelevant question
irrelevant_result = agentic_rag(
    query="What is quantum physics?",
    retriever=retriever,
    llm=llm,
    verbose=True
)

print("\n" + "="*80)
print("📋 FINAL ANSWER:")
print("="*80)
print(irrelevant_result["answer"])

🤖 AGENT STARTING...


🧠 STEP 1: Query Rewriting
--------------------------------------------------------------------------------
Original:  What is quantum physics?
Improved:  What are the fundamental principles and concepts of quantum physics, including its history, key theories (e.g. wave-particle duality, Schrödinger equation), and applications (e.g. quantum computing, quantum cryptography)?

📚 STEP 2: Retrieving Documents
--------------------------------------------------------------------------------
Retrieved 3 chunks

⚖️  STEP 3: Grading Relevance
--------------------------------------------------------------------------------
Agent Decision: NO

⚠️  STEP 4: Agent Declined
--------------------------------------------------------------------------------
Reason: Not enough relevant information

✅ AGENT COMPLETED


📋 FINAL ANSWER:
I do not have enough relevant information in the retrieved documents to answer this question.


---

## Step 14: Compare Simple RAG vs Agentic RAG

Let's see the difference side-by-side!

In [ ]:
test_question = "What is Agentic RAG?"

print("="*80)
print("⚙️  SIMPLE RAG (No Agent)")
print("="*80)
simple_result = simple_rag(test_question, retriever, llm)
print(f"\nAnswer:\n{simple_result['answer']}")

print("\n\n" + "="*80)
print("🤖 AGENTIC RAG (With Agent)")
print("="*80)
agentic_result = agentic_rag(test_question, retriever, llm, verbose=True)

print("\n" + "="*80)
print("📋 FINAL ANSWER:")
print("="*80)
print(agentic_result["answer"])

---

## Step 15: Inspect Retrieved Evidence

See what documents the agent retrieved and judged!

In [ ]:
print("📚 Retrieved Evidence:")
print("="*80)

for i, doc in enumerate(agentic_result["retrieved_docs"], start=1):
    print(f"\n📄 CHUNK {i}:")
    print("-"*80)
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")
    print(f"Page: {doc.metadata.get('page', 'Unknown')}")
    print(f"\nContent Preview:")
    print(doc.page_content[:400] + "...")
    print("-"*80)

---

## Step 16: Try Multiple Questions

Test the agentic system with different types of questions!

In [ ]:
# Test with various questions
test_questions = [
    "How is Agentic RAG different from traditional RAG?",
    "What are the benefits?",
    "tell me about machine learning",  # Vague + possibly irrelevant
    "Explain the key concepts"
]

for q in test_questions:
    print("\n" + "="*80)
    print(f"❓ QUESTION: {q}")
    print("="*80)
    
    result = agentic_rag(q, retriever, llm, verbose=False)
    
    print(f"\n🔄 Improved Query: {result['improved_query']}")
    print(f"⚖️  Relevance: {result['relevance']}")
    print(f"\n📋 Answer:\n{result['answer']}")
    print("="*80)

---

# 🎯 Summary: What We Built

## Simple RAG vs Agentic RAG

### ⚙️ Simple RAG:
```
Question → Retrieve → Answer
```
**Problems:**
- No query improvement
- Answers even with poor context
- Can hallucinate when docs aren't relevant

### 🤖 Agentic RAG:
```
Question → Agent Rewrites → Retrieve → Agent Grades → Agent Answers (if relevant)
```
**Benefits:**
- ✅ Improves vague queries
- ✅ Judges document relevance
- ✅ Declines safely when unsure
- ✅ No hallucinations

---

## Three Agentic Capabilities We Added:

### 1. 🧠 Query Rewriting
- Agent transforms: "tell me stuff" → "What are the main topics discussed?"
- Better retrieval with clearer queries

### 2. ⚖️ Relevance Grading
- Agent judges: Are these docs actually relevant?
- Returns YES/NO decision
- Prevents answering with weak evidence

### 3. 🎯 Grounded Answer Generation
- Agent answers ONLY from retrieved evidence
- No hallucinations
- Safe fallback when evidence is weak

---

## 🆓 100% FREE with Ollama!

No API keys needed - everything runs locally on your computer.

---

## 🚀 Next Steps to Extend This:

You can add more advanced agentic features:
- **Multi-query generation** - Agent generates multiple search queries
- **Self-correction loops** - Agent retries if answer quality is low
- **Tool routing** - Agent chooses between multiple retrieval strategies
- **LangGraph orchestration** - Complex multi-agent workflows
- **ReAct pattern** - Reasoning + Acting in iterative loops

But this is already a **production-ready agentic RAG system**! 🎉